In [1]:
# Change working directory
from pathlib import Path
import os

path = Path(r"P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Beijing")

os.chdir(path)

print(Path.cwd())

P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Beijing


In [2]:
# Load python libraries
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

In [3]:
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

# Load pre-project report
pre_rpt = read_rpt_file("SSF_SDMP_Beijing_pre.rpt")

# Extract data
pre = pre_rpt.node_flooding_summary.copy()

# Ensure Node is index
pre.index.name = "Node"

# Rename columns
pre = pre.rename(columns={
    "Hours_Flooded": "Pre Hours Flooded",
    "Maximum_Rate_CFS": "Pre Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Pre Total Flood Vol (MG)",
})

# Keep only needed columns
pre = pre[[
    "Pre Hours Flooded",
    "Pre Max Flood Rate (cfs)",
    "Pre Total Flood Vol (MG)"
]]

pre.head()

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG)
Node,,,
SQ0001,5.48,2.45,0.022
SQ0002,5.26,4.43,0.020
sQ2177,3.84,3.27,0.018
sQ2177A,2.99,1.93,0.012
sQ2178,2.72,1.05,0.010


In [4]:
# Load post-project report
post_rpt = read_rpt_file("SSF_SDMP_Beijing_imp3.rpt")

# Extract data
post = post_rpt.node_flooding_summary.copy()

# Ensure Node is index
post.index.name = "Node"

# Rename columns
post = post.rename(columns={
    "Hours_Flooded": "Post Hours Flooded",
    "Maximum_Rate_CFS": "Post Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Post Total Flood Vol (MG)",
})

# Keep only needed columns
post = post[[
    "Post Hours Flooded",
    "Post Max Flood Rate (cfs)",
    "Post Total Flood Vol (MG)"
]]

post.head()

,Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG)
Node,,,
SQ0001,2.78,2.89,0.016
SQ0002,2.67,4.88,0.015
sQ2177,2.35,3.04,0.013
sQ2177A,2.03,1.42,0.007
sQ2178,1.92,0.50,0.006


In [5]:
# Merge on index (Node)
comparison = pre.join(post, how="outer")

# Fill missing values
fill_cols = [
    "Pre Total Flood Vol (MG)",
    "Post Total Flood Vol (MG)",
    "Pre Hours Flooded",
    "Post Hours Flooded"
]

for col in fill_cols:
    comparison[col] = comparison[col].fillna(0)

# -----------------------------
# Reduction calculations
# -----------------------------

# Flood volume reduction
comparison["Total Flood Vol Reduction (MG)"] = (
    comparison["Pre Total Flood Vol (MG)"] - comparison["Post Total Flood Vol (MG)"]
)

comparison["Total Flood Vol Percent Reduction"] = np.where(
    comparison["Pre Total Flood Vol (MG)"] > 0,
    (comparison["Total Flood Vol Reduction (MG)"] / comparison["Pre Total Flood Vol (MG)"]) * 100,
    0
)

# Flood duration reduction
comparison["Hours Flooded Reduction"] = (
    comparison["Pre Hours Flooded"] - comparison["Post Hours Flooded"]
)

comparison["Hours Flooded Percent Reduction"] = np.where(
    comparison["Pre Hours Flooded"] > 0,
    (comparison["Hours Flooded Reduction"] / comparison["Pre Hours Flooded"]) * 100,
    0
)

# Round values
comparison["Total Flood Vol Reduction (MG)"] = comparison["Total Flood Vol Reduction (MG)"].round(3)
comparison["Total Flood Vol Percent Reduction"] = comparison["Total Flood Vol Percent Reduction"].round(1)

comparison["Hours Flooded Reduction"] = comparison["Hours Flooded Reduction"].round(2)
comparison["Hours Flooded Percent Reduction"] = comparison["Hours Flooded Percent Reduction"].round(1)

# Sort by severity
comparison = comparison.sort_values(
    by="Pre Total Flood Vol (MG)",
    ascending=False
)

comparison.head(20)

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG),Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG),Total Flood Vol Reduction (MG),Total Flood Vol Percent Reduction,Hours Flooded Reduction,Hours Flooded Percent Reduction
Node,,,,,,,,,,
BeijingPumpStation,3.36,8.69,0.518,0.00,NaN,0.000,0.518,100.0,3.36,100.0
sQ3AC,22.97,11.11,0.068,22.97,11.15,0.066,0.002,2.9,0.00,0.0
sQ3AC1,6.58,7.22,0.063,5.79,7.13,0.061,0.002,3.2,0.79,12.0
sQ3AB,5.13,5.14,0.048,4.20,5.08,0.045,0.003,6.3,0.93,18.1
sQ3AA,4.39,2.48,0.025,2.57,2.74,0.021,0.004,16.0,1.82,41.5
sQ3B,5.04,2.75,0.024,2.61,3.02,0.018,0.006,25.0,2.43,48.2
sQ3C,5.32,2.51,0.023,2.69,2.89,0.017,0.006,26.1,2.63,49.4
sQ3D,5.42,2.53,0.023,2.72,2.98,0.017,0.006,26.1,2.70,49.8
SQ0001,5.48,2.45,0.022,2.78,2.89,0.016,0.006,27.3,2.70,49.3


In [6]:
# Reset index so Node becomes a column
export_df = comparison.reset_index()

# Export to CSV
export_df.to_csv("flood_comparison.csv", index=False)

print("Exported: flood_comparison.csv")

Exported: flood_comparison.csv


In [7]:
# -----------------------------
# Entire network flood reduction
# -----------------------------

# --- Number of flooded nodes ---
pre_flooded_nodes = (comparison["Pre Hours Flooded"] > 0).sum()
post_flooded_nodes = (comparison["Post Hours Flooded"] > 0).sum()
flooded_nodes_reduction = pre_flooded_nodes - post_flooded_nodes
flooded_nodes_percent_reduction = float(np.where(
    pre_flooded_nodes > 0,
    (flooded_nodes_reduction / pre_flooded_nodes) * 100,
    0
))

# --- Average flood duration (flooded nodes only) ---
pre_avg_duration = comparison.loc[comparison["Pre Hours Flooded"] > 0, "Pre Hours Flooded"].mean()
post_avg_duration = comparison.loc[comparison["Post Hours Flooded"] > 0, "Post Hours Flooded"].mean()
pre_avg_duration = pre_avg_duration if not pd.isna(pre_avg_duration) else 0.0
post_avg_duration = post_avg_duration if not pd.isna(post_avg_duration) else 0.0
avg_duration_reduction = pre_avg_duration - post_avg_duration
avg_duration_percent_reduction = float(np.where(
    pre_avg_duration > 0,
    (avg_duration_reduction / pre_avg_duration) * 100,
    0
))

# --- Volume ---
network_pre_total = comparison["Pre Total Flood Vol (MG)"].sum()
network_post_total = comparison["Post Total Flood Vol (MG)"].sum()
network_reduction_mg = network_pre_total - network_post_total

network_percent_reduction = np.where(
    network_pre_total > 0,
    (network_reduction_mg / network_pre_total) * 100,
    0
)

# -----------------------------
# Print results
# -----------------------------

print("--- Flood Volume ---")
print(f"Pre-Project Network Total Flood Volume: {network_pre_total:.3f} MG")
print(f"Post-Project Network Total Flood Volume: {network_post_total:.3f} MG")
print(f"Network Flood Volume Reduction: {network_reduction_mg:.3f} MG")
print(f"Network Flood Volume Percent Reduction: {network_percent_reduction:.1f}%")

print("\n--- Flooded Nodes ---")
print(f"Pre-Project Number of Flooded Nodes: {pre_flooded_nodes}")
print(f"Post-Project Number of Flooded Nodes: {post_flooded_nodes}")
print(f"Flooded Nodes Reduction: {flooded_nodes_reduction}")
print(f"Flooded Nodes Percent Reduction: {flooded_nodes_percent_reduction:.1f}%")

print("\n--- Average Flood Duration (Flooded Nodes Only) ---")
print(f"Pre-Project Avg Flood Duration: {pre_avg_duration:.2f} hrs")
print(f"Post-Project Avg Flood Duration: {post_avg_duration:.2f} hrs")
print(f"Avg Flood Duration Reduction: {avg_duration_reduction:.2f} hrs")
print(f"Avg Flood Duration Percent Reduction: {avg_duration_percent_reduction:.1f}%")


--- Flood Volume ---
Pre-Project Network Total Flood Volume: 1.033 MG
Post-Project Network Total Flood Volume: 0.401 MG
Network Flood Volume Reduction: 0.632 MG
Network Flood Volume Percent Reduction: 61.2%

--- Flooded Nodes ---
Pre-Project Number of Flooded Nodes: 25
Post-Project Number of Flooded Nodes: 22
Flooded Nodes Reduction: 3
Flooded Nodes Percent Reduction: 12.0%

--- Average Flood Duration (Flooded Nodes Only) ---
Pre-Project Avg Flood Duration: 5.29 hrs
Post-Project Avg Flood Duration: 3.57 hrs
Avg Flood Duration Reduction: 1.72 hrs
Avg Flood Duration Percent Reduction: 32.5%


In [8]:
# Fix: ensure these are scalars (not numpy arrays)
network_percent_reduction = float(network_percent_reduction)
network_hours_percent_reduction = float(network_hours_percent_reduction)

# -----------------------------
# Create summary table
# -----------------------------
summary_df = pd.DataFrame({
    "Description": [
        "Pre-Project Number of Flooded Nodes",
        "Post-Project Number of Flooded Nodes",
        "Flooded Nodes Reduction",
        "Flooded Nodes Percent Reduction (%)",
        "Pre-Project Avg Flood Duration - Flooded Nodes Only (hrs)",
        "Post-Project Avg Flood Duration - Flooded Nodes Only (hrs)",
        "Avg Flood Duration Reduction (hrs)",
        "Avg Flood Duration Percent Reduction (%)",
        "Pre-Project Network Total Flood Volume (MG)",
        "Post-Project Network Total Flood Volume (MG)",
        "Network Flood Volume Reduction (MG)",
        "Network Flood Volume Percent Reduction (%)",
        "Pre-Project Total Hours Flooded (hrs)",
        "Post-Project Total Hours Flooded (hrs)",
        "Total Hours Flooded Reduction (hrs)",
        "Hours Flooded Percent Reduction (%)"
    ],
    "Value": [
        int(pre_flooded_nodes),
        int(post_flooded_nodes),
        int(flooded_nodes_reduction),
        f"{round(flooded_nodes_percent_reduction, 1)}%",
        round(pre_avg_duration, 2),
        round(post_avg_duration, 2),
        round(avg_duration_reduction, 2),
        f"{round(avg_duration_percent_reduction, 1)}%",
        round(network_pre_total, 3),
        round(network_post_total, 3),
        round(network_reduction_mg, 3),
        f"{round(network_percent_reduction, 1)}%",
        round(network_pre_hours, 2),
        round(network_post_hours, 2),
        round(network_hours_reduction, 2),
        f"{round(network_hours_percent_reduction, 1)}%"
    ]
})

# -----------------------------
# Export summary CSV
# -----------------------------
summary_df.to_csv("network_flood_summary.csv", index=False)

print("Exported: network_flood_summary.csv")


NameError: name 'network_hours_percent_reduction' is not defined